In [1]:
!pip install -q transformers datasets seqeval accelerate bitsandbytes
!pip install -q sentence-transformers hdbscan umap-learn scikit-learn
!pip install -q faiss-gpu langchain langchain-community langchain-huggingface

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
import json
import gc
import torch
from google.colab import drive
import torch
from torch import nn
import os
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, set_seed, pipeline
from seqeval.metrics import precision_score, recall_score, f1_score
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import hdbscan
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score, davies_bouldin_score
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_44238/2373597416.py:26: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#This is our Training NER code where we have used BanglaBERT by Sagor Sarkar with our own annoted_corpus.csv.
#DO NOT RUN THIS!! Because we have already trained and saved the model in our team Google drive and then uploaded the model when the train dataset is released to run the test data set on Clustering and Qweries
set_seed(42)
df = pd.read_csv("annoted_corpus.csv", keep_default_na=False)
df["token"] = df["token"].astype(str)
sents = df.groupby("note_id", sort=False).agg(tokens=("token", list), tags=("tag", list)).reset_index()
tags = sorted(df["tag"].unique(), key=lambda t: (t != "O", t))
label2id = {t: i for i, t in enumerate(tags)}
id2label = {i: t for t, i in label2id.items()}
train_s, tmp = train_test_split(sents, test_size=0.2, random_state=42)
val_s, test_s = train_test_split(tmp, test_size=0.5, random_state=42)
MODEL="sagorsarker/bangla-bert-base"
tok = AutoTokenizer.from_pretrained(MODEL)
def encode(batch):
    enc= tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=128)
    all_labels = []
    for i, tg in enumerate(batch["tags"]):
        prev, lab = None, []
        for w in enc.word_ids(batch_index=i):
            if w is None:   lab.append(-100)
            elif w != prev: lab.append(label2id[tg[w]])
            else:           lab.append(-100)
            prev= w
        all_labels.append(lab)
    enc["labels"] = all_labels
    return enc
def to_ds(d):
    ds = Dataset.from_dict({"tokens": d.tokens.tolist(), "tags": d.tags.tolist()})
    return ds.map(encode, batched=True, remove_columns=["tokens", "tags"])
train_ds, val_ds, test_ds = to_ds(train_s), to_ds(val_s), to_ds(test_s)
all_train_labels = [label for seq in train_ds["labels"] for label in seq if label != -100]
class_counts = np.bincount(all_train_labels, minlength=len(tags))
class_counts = np.maximum(class_counts, 1)
class_weights = len(all_train_labels) / (len(tags) * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to("cuda" if torch.cuda.is_available() else "cpu")
class WeightedNERTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss
def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    true_l, pred_l = [], []
    for pr, lb in zip(preds, p.label_ids):
        true_l.append([id2label[l] for l in lb if l != -100])
        pred_l.append([id2label[p] for p, l in zip(pr, lb) if l != -100])
    return {"precision": precision_score(true_l, pred_l), "recall": recall_score(true_l, pred_l), "f1": f1_score(true_l, pred_l)}
model= AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(tags), id2label=id2label, label2id=label2id)
args = TrainingArguments(
    output_dir="ner-out", learning_rate=3e-5, num_train_epochs=25,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1", fp16=torch.cuda.is_available(), report_to="none"
)
trainer = WeightedNERTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=DataCollatorForTokenClassification(tok), compute_metrics=compute_metrics)
trainer.train()

In [4]:
os.makedirs("predictions/analysis", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

test_df = pd.read_csv("medbhasha_bengali_clinical_notes(test data).csv", encoding="utf-8-sig")
print(f"Loaded {len(test_df)} test notes.")
"""as we have already trained the NER model that is BANGLABERT of Sagor Sarkar and then we have uploaded the model in google
drive, as the test set released on 10th novemnber and we have trained the model before that in google collab. For this reason, we have uploaded and imported the same model here after the release of this dataset to save time and  cloud GPU ram
-this bangla BERT model is trained with our own annoted_corpus.csv"""
MODEL_PATH= "/content/drive/MyDrive/Pragyan_NER_Final"
ner_tokenizer= AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
ner_model= AutoModelForTokenClassification.from_pretrained(MODEL_PATH)

VALID_ENTITY_TYPES= [
    "DIAGNOSIS", "SYMPTOM", "MEDICATION", "DOSAGE", "TEST",
    "TEST_RESULT", "BODY_SITE", "PROCEDURE", "TEMPORAL", "NEGATION_UNCERTAINTY"
]

ner_pipe= pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=ner_tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    aggregation_strategy="simple"
)

ner_rows = []
for idx, row in test_df.iterrows():
    note_id = str(row["note_id"])
    raw_text = str(row["note_text"]) if pd.notna(row["note_text"]) else ""
    if not raw_text:
        continue

    preds = ner_pipe(raw_text)

    merged_spans = []
    for p in preds:
        ent_type = str(p["entity_group"]).replace("B-", "").replace("I-", "").strip()
        if ent_type not in VALID_ENTITY_TYPES or ent_type == "O":
            continue
        start_idx, end_idx = int(p["start"]), int(p["end"])

        while start_idx < end_idx and raw_text[start_idx].isspace():
            start_idx += 1
        while end_idx > start_idx and raw_text[end_idx - 1].isspace():
            end_idx -= 1
        if start_idx >= end_idx:
            continue

        if merged_spans and merged_spans[-1]["entity_type"] == ent_type and start_idx <= merged_spans[-1]["end"] + 1:
            gap = raw_text[merged_spans[-1]["end"]:start_idx]
            if gap == "" or gap == " ":
                merged_spans[-1]["end"] = end_idx
                continue

        merged_spans.append({"note_id": note_id, "start": start_idx, "end": end_idx, "entity_type": ent_type})
    ner_rows.extend(merged_spans)
ner_pred_df = pd.DataFrame(ner_rows, columns=["note_id", "start", "end", "entity_type"])
ner_pred_df.to_csv("predictions/ner_predictions.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(ner_pred_df)} entity spans to predictions/ner_predictions.csv")

Loaded 1691 test notes.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved 140742 entity spans to predictions/ner_predictions.csv


In [5]:
freq_df= pd.DataFrame(0, index=test_df["note_id"].astype(str), columns=VALID_ENTITY_TYPES)
if not ner_pred_df.empty:
    counts= ner_pred_df.groupby(["note_id", "entity_type"]).size().unstack(fill_value=0)
    for col in VALID_ENTITY_TYPES:
        if col in counts.columns:
            freq_df.loc[counts.index, col] = counts[col]
ner_freq_matrix= freq_df.loc[test_df["note_id"].astype(str)].values
sbert_model= SentenceTransformer("l3cube-pune/bengali-sentence-bert-nli", device="cpu")
text_embeddings = sbert_model.encode(test_df["note_text"].fillna("").astype(str).tolist(), show_progress_bar=True, batch_size=32)

#PCA & K Means
hybrid_features= np.hstack([text_embeddings, ner_freq_matrix])
scaled_features = StandardScaler().fit_transform(hybrid_features)
features_5d = PCA(n_components=5, random_state=42).fit_transform(scaled_features)
cluster_labels = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(features_5d)
print(f"\nREPORT METRICS\n Silhouette Score: {silhouette_score(features_5d, cluster_labels):.4f} \n Davies-Bouldin Index: {davies_bouldin_score(features_5d, cluster_labels):.4f}\n")
pd.DataFrame({"note_id": test_df["note_id"].astype(str), "cluster_id": cluster_labels}).to_csv("predictions/clusters.csv", index=False, encoding="utf-8-sig")
entity_totals= freq_df.sum().sort_values(ascending=False).reset_index()
entity_totals.columns = ["entity_type", "total_count"]
entity_totals.to_csv("predictions/analysis/entity_frequency_summary.csv", index=False, encoding="utf-8-sig")
freq_with_cluster= freq_df.copy()
freq_with_cluster["cluster_id"]= cluster_labels
freq_with_cluster.groupby("cluster_id").mean().round(2).to_csv("predictions/analysis/cluster_entity_profiles.csv", encoding="utf-8-sig")
cooccurrence_matrix= (freq_df > 0).astype(int).T.dot((freq_df > 0).astype(int))
cooccurrence_matrix.to_csv("predictions/analysis/entity_relationship_cooccurrence.csv", encoding="utf-8-sig")

#Visualization
plt.figure(figsize=(8, 6))
plt.scatter(features_5d[:, 0], features_5d[:, 1], c=cluster_labels, cmap="tab10", alpha=0.8, edgecolors="k")
plt.title("Phenotype Clusters (5D PCA+K-Means)")
plt.savefig("predictions/analysis/phenotype_clusters.png", dpi=300)
plt.close()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/53 [00:00<?, ?it/s]


REPORT METRICS
 Silhouette Score: 0.2428 
 Davies-Bouldin Index: 1.2412



In [6]:
import gc
import torch
import pandas as pd
import json
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Clean up all GPU memory before starting
for var_name in ["ner_pipe", "ner_model", "model", "trainer", "llm_model", "llm"]:
    if var_name in locals() or var_name in globals():
        try:
            del globals()[var_name]
        except KeyError:
            pass
        try:
            del locals()[var_name]
        except KeyError:
            pass

gc.collect()
torch.cuda.empty_cache()

test_df = pd.read_csv("medbhasha_bengali_clinical_notes(test data).csv", encoding="utf-8-sig")

#FAISS Retrievers
lc_embeddings= HuggingFaceEmbeddings(model_name="l3cube-pune/bengali-sentence-bert-nli", model_kwargs={"device": "cpu"})
test_docs= [Document(page_content=str(row["note_text"]), metadata={"note_id": str(row["note_id"])}) for _, row in test_df.iterrows() if pd.notna(row["note_text"])]
vectorstore_test = FAISS.from_documents(test_docs, lc_embeddings)

# Correctly reconstruct full texts from the tokenized annoted_corpus.csv
own_df= pd.read_csv("annoted_corpus.csv", encoding="utf-8-sig", keep_default_na=False)
own_df["token"] = own_df["token"].astype(str)

# Group tokens by note_id and join them into full notes
own_sents = own_df.groupby("note_id", sort=False).agg(tokens=("token", list)).reset_index()
own_docs = []
for _, row in own_sents.iterrows():
    full_text = " ".join(row["tokens"])
    if full_text.strip():
        own_docs.append(Document(page_content=full_text, metadata={"note_id": str(row["note_id"])}))

vectorstore_own= FAISS.from_documents(own_docs, lc_embeddings)

#Llama-3- 4-Bit
llm_model= AutoModelForCausalLM.from_pretrained("unsloth/llama-3-8b-Instruct-bnb-4bit", quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16), device_map="auto")
llm_tokenizer= AutoTokenizer.from_pretrained("unsloth/llama-3-8b-Instruct-bnb-4bit", clean_up_tokenization_spaces=False)
llm= HuggingFacePipeline(pipeline=pipeline("text-generation", model=llm_model, tokenizer=llm_tokenizer, max_new_tokens=220, temperature=0.1, repetition_penalty=1.1, return_full_text=False))

prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
আপনি একজন নির্ভুল ক্লিনিকাল এআই (AI) অ্যাসিস্ট্যান্ট। শুধুমাত্র প্রদত্ত ক্লিনিকাল নোটগুলো (Context) ব্যবহার করে বাংলায় প্রশ্নের উত্তর দিন:
১. যদি প্রশ্নে নির্দিষ্ট রোগী খুঁজতে বলা হয়, তবে ম্যাচ করা রোগীদের নোট আইডি এবং প্রত্যেকের এক লাইনের সংক্ষিপ্ত বিবরণ দিন।
২. যদি ওষুধ বা পরীক্ষার তালিকা চাওয়া হয়, তবে পুনরাবৃত্তি না করে তালিকা ও সংখ্যা উল্লেখ করুন।
৩. প্রশ্নের প্রতিটি শর্ত, সংখ্যাগত সীমা এবং নেতিবাচক শর্ত (যেমন: অ্যান্টিবায়োটিক প্রয়োজন নেই) কঠোরভাবে মেনে চলুন।
৪. যদি প্রদত্ত নোটগুলোর মধ্যে প্রশ্নের সাথে মেলে এমন কোনো রোগী বা তথ্য না থাকে, তবে কিছু বানিয়ে না লিখে স্পষ্টভাবে বলুন: "প্রদত্ত নোটে কোনো মিলে যাওয়া রোগী বা তথ্য পাওয়া যায়নি।"<|eot_id|><|start_header_id|>user<|end_header_id|>
ক্লিনিকাল নোটসমূহ (Context):
{context}

প্রশ্ন (Question): {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
llm_chain = PromptTemplate(template=prompt_template, input_variables=["context", "question"]) | llm | StrOutputParser()

queries_df = pd.read_csv("test_queries_bn.csv", encoding="utf-8-sig")
with open("predictions/rag_outputs.jsonl", "w", encoding="utf-8") as f_out:
    for _, q_row in queries_df.iterrows():
        qid, qtext = str(q_row["query_id"]), str(q_row["query_text"])
        for corpus_label, vstore in [("test", vectorstore_test), ("own", vectorstore_own)]:
            # Reduced batch retrieval to prevent OOM
            retrieved_docs = vstore.similarity_search(qtext, k=4)
            context = "\n\n".join(f"[নোট আইডি: {d.metadata['note_id']}] {d.page_content}" for d in retrieved_docs[:3])

            f_out.write(json.dumps({
                "query_id": qid, "corpus": corpus_label,
                "retrieved_note_ids": [d.metadata["note_id"] for d in retrieved_docs],
                "retrieved_passages": [d.page_content for d in retrieved_docs],
                "answer": llm_chain.invoke({"context": context, "question": qtext}).strip()
            }, ensure_ascii=False) + "\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=220) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging